In [0]:
# ============================================================
# 00_permissions_setup — Run ONCE by workspace admin (Shreya)
# ============================================================
# Purpose : Create schemas, grant privileges to all 3 team
#           members so every notebook can read/write tables.
#
# Who runs this : sakhare.c (workspace admin / catalog owner)
# When          : Before running any other notebook
# Re-runnable   : Yes — all statements are idempotent
#
# Azure Databricks Unity Catalog permission model:
#   The catalog owner must grant USE CATALOG, USE SCHEMA,
#   CREATE TABLE, MODIFY, SELECT, and READ/WRITE VOLUME.
#   Without these, team members get 'permission denied' errors
#   when notebooks try to create or write Delta tables.
# ============================================================


In [0]:
# ── Step 1: Detect and verify catalog ───────────────────────
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
if CATALOG in ("hive_metastore", "spark_catalog"):
    catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    project_cats = [c for c in catalogs
                    if c not in ("hive_metastore", "spark_catalog",
                                 "system", "samples", "workspace")]
    CATALOG = project_cats[0] if project_cats else CATALOG

print(f"Detected catalog : {CATALOG}")
print(f"Expected catalog : food_inspection_group8")

if CATALOG != "food_inspection_group8":
    raise Exception(
        f"Wrong catalog '{CATALOG}'. "
        "Switch to 'food_inspection_group8' in the top-right "
        "catalog dropdown in Databricks, then re-run."
    )
print("✓ Correct catalog confirmed")


In [0]:
# ── Step 2: Create schemas ───────────────────────────────────
# IF NOT EXISTS makes this idempotent — safe to re-run
for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{schema}`")
    print(f"✓ Schema ready : {CATALOG}.{schema}")


In [0]:
# ── Step 3: Create Bronze Volume for raw CSV uploads ─────────
# The Volume is the landing zone for raw CSV files.
# After creating it, upload both CSVs via Databricks UI:
#   Catalog Explorer → food_inspection_group8 → bronze
#   → Volumes → raw_data → Upload button
#
# Required filenames (exact):
#   Food_Inspections_20260408.csv
#   Dallas_Inspections_20260408.csv

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`bronze`.`raw_data`
""")

print(f"✓ Volume ready : {CATALOG}.bronze.raw_data")
print()
print("ACTION REQUIRED — upload these two files to the Volume:")
print(f"  /Volumes/{CATALOG}/bronze/raw_data/Food_Inspections_20260408.csv")
print(f"  /Volumes/{CATALOG}/bronze/raw_data/Dallas_Inspections_20260408.csv")


In [0]:
# ── Step 4: Grant privileges to all team members ─────────────
#
# Unity Catalog requires explicit grants at each level:
#   CATALOG → SCHEMA → TABLE/VOLUME
#
# Required grants per member per schema:
#   USE SCHEMA    — allows the member to list objects in schema
#   CREATE TABLE  — allows creating Delta tables
#   MODIFY        — allows INSERT, UPDATE, DELETE, MERGE
#   SELECT        — allows reading table data
#
# ⚠️  UPDATE the MEMBERS list with your actual email addresses.
#     The emails must match the Databricks workspace users exactly.

MEMBERS = [
    "sakhare.c@northeastern.edu",    # admin / catalog owner
    "darban.s@northeastern.edu",     # member
    "patgar.d@northeastern.edu",    # member
]

for member in MEMBERS:
    # Catalog-level USE (required before schema-level grants work)
    spark.sql(f"GRANT USE CATALOG ON CATALOG `{CATALOG}` TO `{member}`")

for schema in ["bronze", "silver", "gold"]:
    for member in MEMBERS:
        spark.sql(f"GRANT USE SCHEMA  ON SCHEMA `{CATALOG}`.`{schema}` TO `{member}`")
        spark.sql(f"GRANT CREATE TABLE ON SCHEMA `{CATALOG}`.`{schema}` TO `{member}`")
        spark.sql(f"GRANT MODIFY       ON SCHEMA `{CATALOG}`.`{schema}` TO `{member}`")
        spark.sql(f"GRANT SELECT       ON SCHEMA `{CATALOG}`.`{schema}` TO `{member}`")

# Volume grants — needed to read CSVs from the Volume in notebooks
for member in MEMBERS:
    spark.sql(f"GRANT READ VOLUME  ON VOLUME `{CATALOG}`.`bronze`.`raw_data` TO `{member}`")
    spark.sql(f"GRANT WRITE VOLUME ON VOLUME `{CATALOG}`.`bronze`.`raw_data` TO `{member}`")

print("✓ All grants applied")
print()
print("Members granted access:")
for m in MEMBERS:
    print(f"  {m}")
print()
print("Schemas covered: bronze, silver, gold")
print("Volume granted : bronze.raw_data (READ + WRITE)")


In [0]:
# ── Step 5: Verify setup ─────────────────────────────────────
print("=== Current grants on bronze schema ===")
display(spark.sql(f"SHOW GRANTS ON SCHEMA `{CATALOG}`.`bronze`"))

print("\n=== Schema inventory ===")
for schema in ["bronze", "silver", "gold"]:
    tables = spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{schema}`").collect()
    vols   = spark.sql(f"SHOW VOLUMES IN `{CATALOG}`.`{schema}`").collect()
    print(f"  {CATALOG}.{schema} — {len(tables)} tables, {len(vols)} volumes")

print("\n=== Volume file check ===")
try:
    files = dbutils.fs.ls(f"/Volumes/{CATALOG}/bronze/raw_data/")
    if files:
        for f in files:
            print(f"  ✓ {f.name}  ({f.size:,} bytes)")
    else:
        print("  Volume is empty — upload the two CSV files now")
except Exception as e:
    print(f"  Volume not yet accessible: {e}")

print()
print("✓ Setup complete. Run order for remaining notebooks:")
print("  1. 01_raw_to_bronze_chicago")
print("  2. 02_raw_to_bronze_dallas")
print("  3. 02_silver_chicago")
print("  4. 03_silver_dallas")
print("  5. 04_silver_unified")
